# Part 3 — Advanced Modeling: Ensembles, Tuning, and Full ML Pipeline
### Loan Approval / Credit Risk Dataset

This notebook picks up the classification task from Part 2 (predicting `Loan_Status`), builds tree-based and
ensemble models, tunes the best one systematically, and serializes it into a reproducible `sklearn` pipeline.


## Setup — Reproducing Part 2's Preprocessing

Same encoding, split, and scaling as Part 2, so `X_train_scaled`, `X_test_scaled`, `y_clf_train`, `y_clf_test`
are identical to what Part 2 used.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline
np.random.seed(42)


In [2]:
df = pd.read_csv('cleaned_data.csv')

y_reg = df['LoanAmount'].copy()
y_clf = (df['Loan_Status'] == 'Y').astype(int)
X = df.drop(columns=['Loan_ID', 'LoanAmount', 'Loan_Status'])

education_order = {'Not Graduate': 0, 'Graduate': 1}
X['Education'] = X['Education'].map(education_order)
nominal_cols = ['Gender', 'Married', 'Self_Employed', 'Property_Area']
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)


Train: (491, 11) Test: (123, 11)


## Task 1 — Decision Tree Baseline (Unconstrained)

In [3]:
dt_unconstrained = DecisionTreeClassifier(max_depth=None, random_state=42)
dt_unconstrained.fit(X_train_scaled, y_clf_train)

train_acc_unc = accuracy_score(y_clf_train, dt_unconstrained.predict(X_train_scaled))
test_acc_unc = accuracy_score(y_clf_test, dt_unconstrained.predict(X_test_scaled))

print(f"Unconstrained Tree — Train accuracy: {train_acc_unc:.4f}, Test accuracy: {test_acc_unc:.4f}")
print(f"Train-test gap: {train_acc_unc - test_acc_unc:.4f}")


Unconstrained Tree — Train accuracy: 1.0000, Test accuracy: 0.7073
Train-test gap: 0.2927


**Finding**: the unconstrained tree's train accuracy is near-perfect while test accuracy is
meaningfully lower — a clear sign of **overfitting**. Decision trees are described as **high-variance**
models because they fit the training data greedily: at each node, the algorithm picks the single best split
for the data present at that node and never revisits or reconsiders earlier splits. Left unconstrained, it
keeps splitting until nodes are pure (or tiny), effectively memorizing noise and idiosyncrasies specific to the
training rows rather than learning patterns that generalize.


## Task 2 — Controlled Decision Tree (max_depth=5, min_samples_split=20)

In [4]:
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_ctrl = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_ctrl = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print(f"Controlled Tree — Train accuracy: {train_acc_ctrl:.4f}, Test accuracy: {test_acc_ctrl:.4f}")
print(f"Train-test gap: {train_acc_ctrl - test_acc_ctrl:.4f}")


Controlled Tree — Train accuracy: 0.8289, Test accuracy: 0.7642
Train-test gap: 0.0647


**`max_depth`** caps how many splits deep any path from root to leaf can go, directly limiting how
complex (and how overfit) the tree's decision boundary can become — a shallower tree trades some bias for
lower variance. **`min_samples_split`** prevents a node from being split further if it holds fewer than 20
samples, which stops the tree from carving out splits that are really just reacting to noise in a handful of
rows. Compared to the unconstrained tree above, the controlled tree's train-test accuracy gap should be
noticeably smaller — the printed gap values above confirm whether that held here.


## Task 3 — Gini vs Entropy (max_depth=5)

In [5]:
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print(f"Gini    — Test accuracy: {test_acc_gini:.4f}")
print(f"Entropy — Test accuracy: {test_acc_entropy:.4f}")


Gini    — Test accuracy: 0.7398
Entropy — Test accuracy: 0.7642


**Gini impurity**: `1 - Σ pᵢ²` (sum over each class's proportion `pᵢ` in the node).
**Entropy**: `-Σ pᵢ log₂(pᵢ)`.

Both measure how "mixed" a node's classes are; a decision tree split is chosen to minimize whichever of these
is used. A node with **Gini = 0** contains samples from only one class — it's perfectly pure, and no split
could improve it further. In practice Gini and Entropy usually produce very similar trees (as the near-identical
test accuracies above show); Gini is slightly cheaper to compute since it avoids the logarithm.


## Task 4 — Random Forest

In [6]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_clf_train)

train_acc_rf = accuracy_score(y_clf_train, rf.predict(X_train_scaled))
test_acc_rf = accuracy_score(y_clf_test, rf.predict(X_test_scaled))
rf_auc = roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:, 1])

print(f"Random Forest — Train accuracy: {train_acc_rf:.4f}, Test accuracy: {test_acc_rf:.4f}, Test AUC: {rf_auc:.4f}")


Random Forest — Train accuracy: 0.9246, Test accuracy: 0.7805, Test AUC: 0.7501


In [7]:
importances = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

top5_features = importances.head(5)
top5_features


,feature,importance
5,Credit_History,0.339896
2,ApplicantIncome,0.235369
3,CoapplicantIncome,0.132386
4,Loan_Amount_Term,0.071701
0,Dependents,0.057709


**How Random Forest computes feature importance**: for each feature, it averages the reduction in Gini
impurity achieved by every split that used that feature, across every tree in the forest. A feature that
consistently produces large, pure splits across many trees gets a high importance score. This differs
fundamentally from a **linear regression coefficient**: a coefficient measures a feature's *linear, additive*
association with the target holding other features fixed, while Random Forest importance measures how much a
feature *actually reduced impurity in practice*, across many non-linear, interaction-aware splits — it can
surface useful non-linear or threshold-based features that a linear coefficient would never highlight.

**Bagging concept**: each tree in the forest is trained on a **bootstrap sample** — a random sample of the
training rows, drawn *with replacement*, so each tree sees a slightly different subset (and some rows
repeated, some left out). At every split, the tree also only considers a **random subset of √(number of
features)** candidate features, rather than all of them. Both of these injections of randomness make individual
trees less correlated with each other. Averaging (or majority-voting) their predictions then cancels out much
of each tree's individual variance/overfitting — the ensemble's prediction is far more stable than any single
deep, unconstrained tree, without needing to sacrifice each tree's depth/flexibility the way Task 2's
`max_depth` constraint did.


## Task 4a — Gradient Boosting

In [8]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_clf_train)

train_acc_gb = accuracy_score(y_clf_train, gb.predict(X_train_scaled))
test_acc_gb = accuracy_score(y_clf_test, gb.predict(X_test_scaled))
gb_auc = roc_auc_score(y_clf_test, gb.predict_proba(X_test_scaled)[:, 1])

print(f"Gradient Boosting — Train accuracy: {train_acc_gb:.4f}, Test accuracy: {test_acc_gb:.4f}, Test AUC: {gb_auc:.4f}")


Gradient Boosting — Train accuracy: 0.8839, Test accuracy: 0.7398, Test AUC: 0.6663


## Task 4b — Feature Ablation Study

Using the Random Forest's `feature_importances_` from Task 4, we identify the 5 lowest-importance features,
remove them, and retrain an identical Random Forest to see whether test-set AUC changes.


In [9]:
lowest5_features = importances.tail(5)['feature'].tolist()
print("5 lowest-importance features:", lowest5_features)

X_train_reduced = X_train_scaled.drop(columns=lowest5_features)
X_test_reduced = X_test_scaled.drop(columns=lowest5_features)

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

auc_full = roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:, 1])
auc_reduced = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print(f"Full-feature Random Forest test AUC:    {auc_full:.4f}")
print(f"Reduced-feature Random Forest test AUC: {auc_reduced:.4f}")
print(f"Difference (reduced - full): {auc_reduced - auc_full:+.4f}")


5 lowest-importance features: ['Married_Yes', 'Education', 'Self_Employed_Yes', 'Gender_Male', 'Property_Area_Urban']


Full-feature Random Forest test AUC:    0.7501
Reduced-feature Random Forest test AUC: 0.7557
Difference (reduced - full): +0.0055


**Finding**: the printed AUC values above show whether removing the 5 lowest-importance features
helped, hurt, or left performance essentially unchanged. If the reduced model's AUC is similar to or higher
than the full model's, those 5 features were genuinely close to noise and can be dropped safely. If AUC drops
meaningfully, they were contributing real (if modest) signal despite their low individual importance scores.

**Production trade-off**: a lower-dimensional model is cheaper to run at inference time, simpler to monitor,
and has fewer upstream data dependencies to maintain (one less feature pipeline that can break). That's a real
engineering win — but it's only worth taking if the AUC degradation is small enough to be within the business's
tolerance for reduced accuracy. Given the AUC difference measured above, we'd only recommend deploying the
reduced model if that gap is negligible relative to the full model's overall AUC.


## Task 5 — Cross-Validated Comparison

5-fold stratified cross-validation (on the training set) for Logistic Regression, the controlled Decision Tree,
Random Forest, and Gradient Boosting, scored on ROC-AUC.


In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=42),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

cv_results = []
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring='roc_auc')
    cv_results.append({'Model': name, 'CV Mean AUC': scores.mean(), 'CV Std AUC': scores.std()})

cv_results_df = pd.DataFrame(cv_results)
cv_results_df


,Model,CV Mean AUC,CV Std AUC
0,Logistic Regression,0.758572,0.034528
1,Decision Tree (depth=5),0.722927,0.048632
2,Random Forest,0.771740,0.056103
3,Gradient Boosting,0.747429,0.044564


**Why cross-validation is more reliable than a single train-test split**: a single split gives one
estimate of generalization performance that depends heavily on which rows happened to land in the test set —
with only 123 test rows here, a few unlucky/lucky rows can swing the metric noticeably. 5-fold CV instead
trains and evaluates the model 5 times on 5 different train/validation partitions of the training data and
reports the mean and spread, giving both a more stable point estimate and a sense of how much that estimate
varies (the standard deviation) — exactly the kind of variability the bootstrap analysis in Part 2 was also
trying to quantify.


## Task 6 — Hyperparameter Tuning with GridSearchCV

In [11]:
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
)
grid_search.fit(X_train, y_clf_train)  # unscaled — the pipeline handles scaling internally

print("Best params:", grid_search.best_params_)
print("Best CV score (ROC-AUC):", round(grid_search.best_score_, 4))

best_pipeline = grid_search.best_estimator_


Best params: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 100}
Best CV score (ROC-AUC): 0.7714


In [12]:
n_configs = 1
for v in param_grid.values():
    n_configs *= len(v)
n_folds = 5
print(f"Total configurations evaluated: {n_configs} param combos x {n_folds} folds = {n_configs * n_folds} model fits")


Total configurations evaluated: 18 param combos x 5 folds = 90 model fits


**Grid Search vs Randomized Search trade-off**: Grid Search exhaustively evaluates every combination in
the grid above, which guarantees finding the best combination *within that grid*, but the number of fits grows
multiplicatively with the number of hyperparameters and values tried — it becomes expensive fast for larger
grids. Randomized Search instead samples a fixed number of random combinations from the specified distributions,
which scales far better to large search spaces and often finds a comparably good configuration for a fraction
of the compute, at the cost of no longer guaranteeing the single best combination is checked.


## Task 7 — Manual Learning Curve

Using the best pipeline from Task 6, we re-fit it on progressively larger slices of `X_train` (unscaled — the
pipeline scales internally) and track training AUC vs. a fixed test AUC.


In [13]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_rows = []

for frac in fractions:
    n_rows = int(frac * len(X_train))
    X_subset = X_train.iloc[:n_rows]
    y_subset = y_clf_train.iloc[:n_rows]

    pipeline_frac = make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        RandomForestClassifier(**{k.split('__')[1]: v for k, v in grid_search.best_params_.items()}, random_state=42)
    )
    pipeline_frac.fit(X_subset, y_subset)

    train_auc = roc_auc_score(y_subset, pipeline_frac.predict_proba(X_subset)[:, 1])
    test_auc = roc_auc_score(y_clf_test, pipeline_frac.predict_proba(X_test)[:, 1])

    learning_curve_rows.append({
        'Training fraction': frac,
        'Training AUC': round(train_auc, 4),
        'Test AUC': round(test_auc, 4)
    })

learning_curve_df = pd.DataFrame(learning_curve_rows)
learning_curve_df


,Training fraction,Training AUC,Test AUC
0,0.2,1.0000,0.7701
1,0.4,1.0000,0.7788
2,0.6,0.9994,0.7526
3,0.8,0.9977,0.7881
4,1.0,0.9956,0.7501


**Interpretation** (read directly off the table above):
(i) if training AUC decreases as the training set grows, that's expected for a high-variance model like Random
Forest — with very few rows it can memorize the training set almost perfectly (training AUC near 1.0), and that
advantage erodes as more, more-varied rows are added.
(ii) if test AUC rises as more training data is added, that tells us this model is currently **data-limited** —
collecting more labeled applications would likely keep improving it.
(iii) if test AUC has instead plateaued by 100% of the training data (flat over the last couple of fractions),
the model is **capacity-limited** on the *current feature set* — more rows of the same 11 features wouldn't
help much, and further gains would need to come from better/more features rather than more data.


## Task 8 — Serialize the Best Model

In [14]:
joblib.dump(best_pipeline, 'best_model.pkl')
print("Saved best_model.pkl")


Saved best_model.pkl


In [15]:
# Reload and predict on two hand-crafted test rows
loaded_pipeline = joblib.load('best_model.pkl')

hand_crafted_rows = pd.DataFrame([
    {  # profile 1: strong applicant
        'Dependents': 0, 'Education': 1, 'ApplicantIncome': 6000, 'CoapplicantIncome': 2000,
        'Loan_Amount_Term': 360, 'Credit_History': 1.0, 'Gender_Male': True, 'Married_Yes': True,
        'Self_Employed_Yes': False, 'Property_Area_Semiurban': True, 'Property_Area_Urban': False
    },
    {  # profile 2: weaker applicant
        'Dependents': 3, 'Education': 0, 'ApplicantIncome': 1800, 'CoapplicantIncome': 0,
        'Loan_Amount_Term': 180, 'Credit_History': 0.0, 'Gender_Male': False, 'Married_Yes': False,
        'Self_Employed_Yes': True, 'Property_Area_Semiurban': False, 'Property_Area_Urban': True
    }
], columns=X_train.columns)

predictions = loaded_pipeline.predict(hand_crafted_rows)
probabilities = loaded_pipeline.predict_proba(hand_crafted_rows)[:, 1]

for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"Profile {i+1}: predicted Loan_Status={'Approved' if pred == 1 else 'Not Approved'}, "
          f"probability of approval={prob:.3f}")


Profile 1: predicted Loan_Status=Approved, probability of approval=0.753
Profile 2: predicted Loan_Status=Not Approved, probability of approval=0.110


## Task 9 — Summary Comparison Table

In [16]:
test_auc_values = {
    'Logistic Regression': roc_auc_score(y_clf_test, LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=42).fit(X_train_scaled, y_clf_train).predict_proba(X_test_scaled)[:, 1]),
    'Decision Tree (depth=5)': roc_auc_score(y_clf_test, dt_controlled.predict_proba(X_test_scaled)[:, 1]),
    'Random Forest': rf_auc,
    'Gradient Boosting': gb_auc,
    'Tuned Random Forest (GridSearchCV)': roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test)[:, 1])
}

summary_table = cv_results_df.copy()
tuned_row = pd.DataFrame([{
    'Model': 'Tuned Random Forest (GridSearchCV)',
    'CV Mean AUC': grid_search.best_score_,
    'CV Std AUC': grid_search.cv_results_['std_test_score'][grid_search.best_index_]
}])
summary_table = pd.concat([summary_table, tuned_row], ignore_index=True)
summary_table['Test AUC'] = summary_table['Model'].map(test_auc_values)
summary_table = summary_table.round(4)
summary_table


,Model,CV Mean AUC,CV Std AUC,Test AUC
0,Logistic Regression,0.7586,0.0345,0.7331
1,Decision Tree (depth=5),0.7229,0.0486,0.6901
2,Random Forest,0.7717,0.0561,0.7501
3,Gradient Boosting,0.7474,0.0446,0.6663
4,Tuned Random Forest (GridSearchCV),0.7714,0.0563,0.7501


**Recommendation**: based on the table above, we recommend the model with the best combination of
**high test-set AUC** and a **CV mean AUC that isn't meaningfully different from its test AUC** (a large gap
between CV and test performance would suggest the test-set number is partly luck). Between the untuned
ensembles and the GridSearchCV-tuned Random Forest, the tuned model is the safer production choice: its
hyperparameters were selected using cross-validation on the training set alone, its test AUC was never used to
pick that configuration, and it's packaged as a single reusable `sklearn` `Pipeline` (imputation → scaling →
model) that can be deployed and reloaded exactly as scored here, which the untuned standalone models are not.
